Подключаем pandas для работы с таблицами и numpy для числовых операций.

In [1]:
import pandas as pd
import numpy as np

Настройка, чтобы pandas показывал все нужные столбцы

In [2]:
pd.set_option('display.max_columns', None)

Загружаем файл titanic.csv

In [3]:
df = pd.read_csv("titanic.csv")

Проверяем, что данные загрузились

In [5]:
print("Размер датасета:", df.shape)

Размер датасета: (891, 12)


 Смотрим на структуру таблицы — какие есть столбцы и как выглядят значения.

In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Определяем признаки и целевую переменную

целевая переменная (target) — то, что модель должна предсказывать. В нашей задаче это Survived (выжил пассажир или нет). Обозначается y

In [7]:
y = df["Survived"]

In [8]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

Выбираем признаки

In [9]:
X = df[["Pclass", "Age", "Sex", "Fare", "SibSp", "Parch"]]

In [10]:
X.head()

,Pclass,Age,Sex,Fare,SibSp,Parch
0,3,22.0,male,7.2500,1,0
1,1,38.0,female,71.2833,1,0
2,3,26.0,female,7.9250,0,0
3,1,35.0,female,53.1000,1,0
4,3,35.0,male,8.0500,0,0


Проверка пропущенных значений

isnull() показывает, где есть пропуски (NaN), а .sum() считает их количество по каждому столбцу.

In [11]:
X.isnull()

,Pclass,Age,Sex,Fare,SibSp,Parch
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
...,...,...,...,...,...,...
886,False,False,False,False,False,False
887,False,False,False,False,False,False
888,False,True,False,False,False,False
889,False,False,False,False,False,False


In [12]:
X.isnull().sum()

Pclass      0
Age       177
Sex         0
Fare        0
SibSp       0
Parch       0
dtype: int64

Обработка пропусков в Age.  в Age есть пропуски. Заменяем их медианой — это устойчивая к выбросам характеристика середины ряда.

Считаем медиану возраста

In [13]:
median_age = X["Age"].median()
print("Медиана Age:", median_age)

Медиана Age: 28.0


Заполняем пропуски медианой

In [16]:
X["Age"] = X["Age"].fillna(median_age)
X.isnull().sum()

C:\Users\Сабрина\AppData\Local\Temp\ipykernel_3812\2415169531.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Age"] = X["Age"].fillna(median_age)


Pclass    0
Age       0
Sex       0
Fare      0
SibSp     0
Parch     0
dtype: int64

Преобразование категориального признака Sex в числовой вид

Преобразуем Sex: male -> 0, female -> 1

In [17]:
X["Sex"] = X["Sex"].map({"male": 0, "female": 1})
X.head()

C:\Users\Сабрина\AppData\Local\Temp\ipykernel_3812\1790464403.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Sex"] = X["Sex"].map({"male": 0, "female": 1})


,Pclass,Age,Sex,Fare,SibSp,Parch
0,3,22.0,0,7.2500,1,0
1,1,38.0,1,71.2833,1,0
2,3,26.0,1,7.9250,0,0
3,1,35.0,1,53.1000,1,0
4,3,35.0,0,8.0500,0,0


Проверка подготовленных данных

info() показывает типы данных и количество непустых значений. Убеждаемся, что все признаки числовые и пропусков нет.

In [18]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Pclass  891 non-null    int64  
 1   Age     891 non-null    float64
 2   Sex     891 non-null    int64  
 3   Fare    891 non-null    float64
 4   SibSp   891 non-null    int64  
 5   Parch   891 non-null    int64  
dtypes: float64(2), int64(4)
memory usage: 41.9 KB


In [19]:
X.isnull().sum()

Pclass    0
Age       0
Sex       0
Fare      0
SibSp     0
Parch     0
dtype: int64

Разделение на обучающую и тестовую выборки

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

 Проверка размеров выборок

In [21]:
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (712, 6)
X_test:  (179, 6)
y_train: (712,)
y_test:  (179,)


Масштабирование признаков

In [23]:
from sklearn.preprocessing import StandardScaler

In [24]:
scaler = StandardScaler()

Для обучающей выборки: fit + transform

In [25]:
X_train_scaled = scaler.fit_transform(X_train)

Для тестовой выборки: только transform

In [26]:
X_test_scaled = scaler.transform(X_test)

Финальная проверка размеров после масштабирования

In [27]:
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled: ", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train_scaled: (712, 6)
X_test_scaled:  (179, 6)
y_train: (712,)
y_test:  (179,)


Первые 5 строк масштабированных обучающих данных

In [28]:
pd.DataFrame(X_train_scaled, columns=X.columns).head()

,Pclass,Age,Sex,Fare,SibSp,Parch
0,-1.614136,1.253641,-0.724310,-0.078684,-0.470722,-0.479342
1,-0.400551,-0.477284,-0.724310,-0.377145,-0.470722,-0.479342
2,0.813034,0.215086,-0.724310,-0.474867,-0.470722,-0.479342
3,0.813034,-0.246494,-0.724310,-0.476230,0.379923,-0.479342
4,0.813034,-1.785093,1.380624,-0.025249,2.931860,2.048742


 Проверка: среднее и стандартное отклонение по обучающей выборке

In [30]:
pd.DataFrame(X_train_scaled, columns=X.columns).agg(["mean", "std"]).round(2)

,Pclass,Age,Sex,Fare,SibSp,Parch
mean,0.0,0.0,0.0,0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0
